In [1]:
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


load_dotenv()

PROJECT_ROOT = Path(r"C:\Users\Playdata\workspace\SKN28-third-2TEAM")
CHROMA_DIR = PROJECT_ROOT / "data" / "vectorstore" / "chroma_db"
COLLECTION_NAME = "kaist_graduate_info"

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),
)

print("CHROMA_DIR:", CHROMA_DIR)
print("CHROMA_DIR exists:", CHROMA_DIR.exists())

# Chroma collection 문서 수 확인
try:
    count = vectorstore._collection.count()
    print("Collection document count:", count)
except Exception as e:
    print("Collection count 확인 실패:", e)


# 1. filter 없이 검색
print("\n" + "=" * 100)
print("[TEST 1] filter 없이 검색")

docs = vectorstore.similarity_search_with_score(
    query="AI컴퓨팅학과 석사 지원 자격",
    k=5,
)

print("검색 결과 수:", len(docs))

for i, (doc, score) in enumerate(docs, start=1):
    print("-" * 80)
    print(f"[결과 {i}] score={score}")
    print(doc.metadata)
    print(doc.page_content[:300])


# 2. dept만 filter
print("\n" + "=" * 100)
print("[TEST 2] dept만 filter")

docs = vectorstore.similarity_search_with_score(
    query="AI컴퓨팅학과 석사 지원 자격",
    k=5,
    filter={"dept": {"$eq": "aic"}},
)

print("검색 결과 수:", len(docs))

for i, (doc, score) in enumerate(docs, start=1):
    print("-" * 80)
    print(f"[결과 {i}] score={score}")
    print(doc.metadata)
    print(doc.page_content[:300])


# 3. content_type만 filter
print("\n" + "=" * 100)
print("[TEST 3] content_type만 filter")

docs = vectorstore.similarity_search_with_score(
    query="AI컴퓨팅학과 석사 지원 자격",
    k=5,
    filter={"content_type": {"$eq": "admission"}},
)

print("검색 결과 수:", len(docs))

for i, (doc, score) in enumerate(docs, start=1):
    print("-" * 80)
    print(f"[결과 {i}] score={score}")
    print(doc.metadata)
    print(doc.page_content[:300])


# 4. dept + content_type filter
print("\n" + "=" * 100)
print("[TEST 4] dept + content_type filter")

docs = vectorstore.similarity_search_with_score(
    query="AI컴퓨팅학과 석사 지원 자격",
    k=5,
    filter={
        "$and": [
            {"dept": {"$eq": "aic"}},
            {"content_type": {"$eq": "admission"}},
        ]
    },
)

print("검색 결과 수:", len(docs))

for i, (doc, score) in enumerate(docs, start=1):
    print("-" * 80)
    print(f"[결과 {i}] score={score}")
    print(doc.metadata)
    print(doc.page_content[:300])

CHROMA_DIR: C:\Users\Playdata\workspace\SKN28-third-2TEAM\data\vectorstore\chroma_db
CHROMA_DIR exists: True
Collection document count: 0

[TEST 1] filter 없이 검색
검색 결과 수: 0

[TEST 2] dept만 filter
검색 결과 수: 0

[TEST 3] content_type만 filter
검색 결과 수: 0

[TEST 4] dept + content_type filter
검색 결과 수: 0


In [2]:
from pathlib import Path
import chromadb

CHROMA_DIR = Path(r"C:\Users\Playdata\workspace\SKN28-third-2TEAM\data\vectorstore\chroma_db")

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

collections = client.list_collections()

print("CHROMA_DIR:", CHROMA_DIR)
print("collection 개수:", len(collections))

for collection in collections:
    print("-" * 80)
    print("name:", collection.name)
    print("count:", collection.count())

CHROMA_DIR: C:\Users\Playdata\workspace\SKN28-third-2TEAM\data\vectorstore\chroma_db
collection 개수: 1
--------------------------------------------------------------------------------
name: kaist_graduate_info
count: 0


In [3]:
from pathlib import Path
import chromadb

PROJECT_ROOT = Path(r"C:\Users\Playdata\workspace\SKN28-third-2TEAM")

candidate_dirs = []

for path in PROJECT_ROOT.rglob("*"):
    if path.is_dir():
        sqlite_file = path / "chroma.sqlite3"
        if sqlite_file.exists():
            candidate_dirs.append(path)

print("찾은 Chroma 후보 폴더 수:", len(candidate_dirs))

for chroma_dir in candidate_dirs:
    print("=" * 100)
    print("CHROMA_DIR:", chroma_dir)

    try:
        client = chromadb.PersistentClient(path=str(chroma_dir))
        collections = client.list_collections()

        print("collection 개수:", len(collections))

        for collection in collections:
            print(" -", collection.name, "count:", collection.count())

    except Exception as e:
        print("확인 실패:", e)

찾은 Chroma 후보 폴더 수: 1
CHROMA_DIR: C:\Users\Playdata\workspace\SKN28-third-2TEAM\data\vectorstore\chroma_db
collection 개수: 1
 - kaist_graduate_info count: 0


In [4]:
from pathlib import Path
import json

PROJECT_ROOT = Path(r"C:\Users\Playdata\workspace\SKN28-third-2TEAM")
JSONL_PATH = PROJECT_ROOT / "data" / "processed" / "json" / "vector_documents.jsonl"

print("JSONL_PATH:", JSONL_PATH)
print("exists:", JSONL_PATH.exists())

if JSONL_PATH.exists():
    with open(JSONL_PATH, "r", encoding="utf-8") as f:
        lines = f.readlines()

    print("line count:", len(lines))

    if lines:
        sample = json.loads(lines[0])
        print("sample keys:", sample.keys())
        print("sample text:", sample.get("text", "")[:300])
        print("sample metadata:", sample.get("metadata", {}))

JSONL_PATH: C:\Users\Playdata\workspace\SKN28-third-2TEAM\data\processed\json\vector_documents.jsonl
exists: True
line count: 699
sample keys: dict_keys(['id', 'text', 'metadata'])
sample text: [AI컴퓨팅학과 입학 정보]
항목: general_info
페이지 제목: 학사과정 입학 안내
섹션: 입학처 홈
제목: 학부 (한국인)
내용: 한국인 학부 신입생 모집 안내입니다.
출처: https://aic.kaist.ac.kr/#/admission-ug
sample metadata: {'source_type': 'csv_admission', 'content_type': 'admission', 'dept': 'aic', 'dept_name': 'AI컴퓨팅학과', 'title': '학부 (한국인)', 'section': '입학처 홈', 'admission_type': 'general_info', 'schedule_date': None, 'source_url': 'https://aic.kaist.ac.kr/#/admission-ug', 'crawled_at': '2026-05-26T13:24:44.960971+00:00'}


In [1]:
"""
build_vectorstore.py

vector_documents.jsonl 파일을 읽어서 Chroma vectorstore를 생성합니다.

현재 프로젝트 구조:
C:.
├─ data
│  ├─ processed
│  │  ├─ csv
│  │  ├─ json
│  │  │  └─ vector_documents.jsonl
│  │  └─ reports
│  ├─ raw_data
│  └─ vectorstore
│      └─ chroma_db
"""

from __future__ import annotations

import hashlib
import json
import re
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from tqdm import tqdm

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


# ============================================================
# 1. 설정
# ============================================================

@dataclass
class VectorStoreBuildConfig:
    project_root: Path = Path(r"C:\Users\Playdata\workspace\SKN28-third-2TEAM")

    jsonl_relative_path: Path = Path("data") / "processed" / "json" / "vector_documents.jsonl"

    chroma_relative_dir: Path = Path("data") / "vectorstore" / "chroma_db"

    collection_name: str = "kaist_graduate_info"

    embedding_model: str = "text-embedding-3-small"

    batch_size: int = 100

    reset: bool = True

    @property
    def jsonl_path(self) -> Path:
        return self.project_root / self.jsonl_relative_path

    @property
    def chroma_dir(self) -> Path:
        return self.project_root / self.chroma_relative_dir


# ============================================================
# 2. 텍스트 / metadata 정리
# ============================================================

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def safe_metadata_value(value: Any) -> Any:
    if value is None:
        return None

    if isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)

    return str(value)


def normalize_metadata(raw_metadata: dict[str, Any] | None) -> dict[str, Any]:
    metadata: dict[str, Any] = {}

    if not isinstance(raw_metadata, dict):
        raw_metadata = {}

    for key, value in raw_metadata.items():
        safe_value = safe_metadata_value(value)

        if safe_value is not None:
            metadata[key] = safe_value

    # alias 통일
    if "dept_name" in metadata:
        metadata["department"] = metadata["dept_name"]

    if "dept" in metadata:
        metadata["department_code"] = metadata["dept"]

    if "content_type" in metadata:
        metadata["doc_type"] = metadata["content_type"]

    if "source_url" in metadata:
        metadata["source"] = metadata["source_url"]
    elif "url" in metadata:
        metadata["source"] = metadata["url"]
    elif "source" not in metadata:
        metadata["source"] = "unknown"

    return metadata


def build_page_content(text: str, metadata: dict[str, Any]) -> str:
    """
    검색 효율을 위해 핵심 metadata를 page_content 앞에 붙입니다.
    """
    keyword_lines: list[str] = []

    field_map = [
        ("dept_name", "학과"),
        ("content_type", "문서유형"),
        ("source_type", "데이터출처유형"),
        ("title", "제목"),
        ("section", "섹션"),
        ("admission_type", "입학항목"),
        ("course_code", "과목코드"),
        ("course_code_norm", "정규화 과목코드"),
        ("course_level", "과목수준"),
        ("course_type", "이수구분"),
        ("tracks", "관련트랙"),
        ("name", "이름"),
        ("role", "역할"),
        ("email", "이메일"),
        ("phone", "전화번호"),
        ("website", "웹사이트"),
        ("event_date", "행사일"),
        ("url", "URL"),
        ("source_url", "출처URL"),
    ]

    for key, label in field_map:
        value = metadata.get(key)

        if value:
            keyword_lines.append(f"{label}: {value}")

    if keyword_lines:
        keyword_header = "\n".join(keyword_lines)

        return f"[검색 키워드]\n{keyword_header}\n\n[문서 내용]\n{text}"

    return text


def make_content_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def make_document_id(
    original_id: str | None,
    page_content: str,
    metadata: dict[str, Any],
) -> str:
    if original_id:
        return str(original_id)

    base = "|".join(
        [
            str(metadata.get("dept", "")),
            str(metadata.get("content_type", "")),
            str(metadata.get("title", "")),
            page_content,
        ]
    )

    return hashlib.sha256(base.encode("utf-8")).hexdigest()


# ============================================================
# 3. JSONL → Document 변환
# ============================================================

def load_documents_from_jsonl(
    jsonl_path: Path,
) -> tuple[list[Document], list[str]]:
    if not jsonl_path.exists():
        raise FileNotFoundError(f"JSONL 파일을 찾을 수 없습니다: {jsonl_path}")

    documents: list[Document] = []
    ids: list[str] = []

    seen_content_hashes: set[str] = set()
    seen_ids: set[str] = set()

    parse_error_count = 0
    empty_text_count = 0
    duplicate_count = 0

    with open(jsonl_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    print("[JSONL 확인]")
    print("path:", jsonl_path)
    print("line count:", len(lines))

    for line_number, line in enumerate(tqdm(lines, desc="JSONL 로드"), start=1):
        line = line.strip()

        if not line:
            continue

        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            parse_error_count += 1
            continue

        text = clean_text(item.get("text", ""))

        if not text:
            empty_text_count += 1
            continue

        metadata = normalize_metadata(item.get("metadata", {}))

        page_content = build_page_content(
            text=text,
            metadata=metadata,
        )

        content_hash = make_content_hash(page_content)

        if content_hash in seen_content_hashes:
            duplicate_count += 1
            continue

        seen_content_hashes.add(content_hash)

        metadata["content_hash"] = content_hash
        metadata["char_len"] = len(page_content)

        original_id = item.get("id")
        doc_id = make_document_id(
            original_id=original_id,
            page_content=page_content,
            metadata=metadata,
        )

        if doc_id in seen_ids:
            doc_id = make_content_hash(f"{doc_id}-{line_number}-{page_content}")

        seen_ids.add(doc_id)

        metadata["original_id"] = str(original_id or doc_id)

        document = Document(
            page_content=page_content,
            metadata=metadata,
        )

        documents.append(document)
        ids.append(doc_id)

    print("\n[Document 변환 결과]")
    print("documents:", len(documents))
    print("ids:", len(ids))
    print("parse_error_count:", parse_error_count)
    print("empty_text_count:", empty_text_count)
    print("duplicate_count:", duplicate_count)

    if documents:
        print("\n[첫 번째 Document 확인]")
        print("metadata:", documents[0].metadata)
        print("content:", documents[0].page_content[:500])

    return documents, ids


# ============================================================
# 4. Chroma 생성
# ============================================================

def build_vectorstore(
    config: VectorStoreBuildConfig,
) -> Chroma:
    load_dotenv()

    if config.reset and config.chroma_dir.exists():
        print("[초기화] 기존 Chroma DB 삭제:", config.chroma_dir)
        shutil.rmtree(config.chroma_dir)

    config.chroma_dir.mkdir(parents=True, exist_ok=True)

    documents, ids = load_documents_from_jsonl(config.jsonl_path)

    if not documents:
        raise ValueError("저장할 Document가 없습니다. JSONL 파일을 확인하세요.")

    embedding_model = OpenAIEmbeddings(
        model=config.embedding_model,
    )

    vectorstore = Chroma(
        collection_name=config.collection_name,
        embedding_function=embedding_model,
        persist_directory=str(config.chroma_dir),
    )

    print("\n[Chroma 인덱싱 시작]")
    print("collection_name:", config.collection_name)
    print("chroma_dir:", config.chroma_dir)
    print("총 문서 수:", len(documents))

    for start in tqdm(range(0, len(documents), config.batch_size), desc="Chroma 저장"):
        end = start + config.batch_size

        batch_documents = documents[start:end]
        batch_ids = ids[start:end]

        vectorstore.add_documents(
            documents=batch_documents,
            ids=batch_ids,
        )

    count = vectorstore._collection.count()

    print("\n[Chroma 저장 완료]")
    print("collection count:", count)

    if count == 0:
        raise RuntimeError(
            "Chroma collection count가 0입니다. 저장이 실패했습니다."
        )

    return vectorstore


# ============================================================
# 5. 저장 확인
# ============================================================

def run_smoke_test(
    vectorstore: Chroma,
) -> None:
    print("\n[Smoke Test]")

    test_queries = [
        "AI컴퓨팅학과 석사 지원 자격",
        "AI컴퓨팅학과 학과설명회",
        "AX학과 교수 이메일",
    ]

    for query in test_queries:
        print("=" * 100)
        print("query:", query)

        results = vectorstore.similarity_search_with_score(
            query=query,
            k=3,
        )

        print("result count:", len(results))

        for index, (document, score) in enumerate(results, start=1):
            print("-" * 80)
            print(f"[{index}] score:", score)
            print("metadata:", document.metadata)
            print("content:", document.page_content[:300])


def main() -> None:
    config = VectorStoreBuildConfig(
        reset=True,
    )

    print("[설정]")
    print("project_root:", config.project_root)
    print("jsonl_path:", config.jsonl_path)
    print("chroma_dir:", config.chroma_dir)
    print("collection_name:", config.collection_name)

    vectorstore = build_vectorstore(config)
    run_smoke_test(vectorstore)


if __name__ == "__main__":
    main()

[설정]
project_root: C:\Users\Playdata\workspace\SKN28-third-2TEAM
jsonl_path: C:\Users\Playdata\workspace\SKN28-third-2TEAM\data\processed\json\vector_documents.jsonl
chroma_dir: C:\Users\Playdata\workspace\SKN28-third-2TEAM\data\vectorstore\chroma_db
collection_name: kaist_graduate_info
[초기화] 기존 Chroma DB 삭제: C:\Users\Playdata\workspace\SKN28-third-2TEAM\data\vectorstore\chroma_db
[JSONL 확인]
path: C:\Users\Playdata\workspace\SKN28-third-2TEAM\data\processed\json\vector_documents.jsonl
line count: 699


JSONL 로드: 100%|██████████| 699/699 [00:00<00:00, 48006.72it/s]


[Document 변환 결과]
documents: 699
ids: 699
parse_error_count: 0
empty_text_count: 0
duplicate_count: 0

[첫 번째 Document 확인]
metadata: {'source_type': 'csv_admission', 'content_type': 'admission', 'dept': 'aic', 'dept_name': 'AI컴퓨팅학과', 'title': '학부 (한국인)', 'section': '입학처 홈', 'admission_type': 'general_info', 'source_url': 'https://aic.kaist.ac.kr/#/admission-ug', 'crawled_at': '2026-05-26T13:24:44.960971+00:00', 'department': 'AI컴퓨팅학과', 'department_code': 'aic', 'doc_type': 'admission', 'source': 'https://aic.kaist.ac.kr/#/admission-ug', 'content_hash': 'c28475f865a9e96b7d15b25f0c323020dd08fd1b5312a66902ccd392961c62b0', 'char_len': 299, 'original_id': '17054ead3800d21052c9'}
content: [검색 키워드]
학과: AI컴퓨팅학과
문서유형: admission
데이터출처유형: csv_admission
제목: 학부 (한국인)
섹션: 입학처 홈
입학항목: general_info
출처URL: https://aic.kaist.ac.kr/#/admission-ug

[문서 내용]
[AI컴퓨팅학과 입학 정보]
항목: general_info
페이지 제목: 학사과정 입학 안내
섹션: 입학처 홈
제목: 학부 (한국인)
내용: 한국인 학부 신입생 모집 안내입니다.
출처: https://aic.kaist.ac.kr/#/admission-ug



[Chroma 인덱싱 시작]
collection_name: kaist_graduate_info
chroma_dir: C:\Users\Playdata\workspace\SKN28-third-2TEAM\data\vectorstore\chroma_db
총 문서 수: 699


Chroma 저장: 100%|██████████| 7/7 [00:06<00:00,  1.10it/s]



[Chroma 저장 완료]
collection count: 699

[Smoke Test]
query: AI컴퓨팅학과 석사 지원 자격
result count: 3
--------------------------------------------------------------------------------
[1] score: 0.8486237525939941
metadata: {'content_type': 'admission', 'source': 'https://aic.kaist.ac.kr/#/admission-grad', 'department': 'AI컴퓨팅학과', 'source_url': 'https://aic.kaist.ac.kr/#/admission-grad', 'dept': 'aic', 'department_code': 'aic', 'doc_type': 'admission', 'dept_name': 'AI컴퓨팅학과', 'section': '지원 자격', 'title': '석사과정', 'original_id': '7ec64c3fd9661c29ad86', 'admission_type': 'eligibility', 'char_len': 324, 'crawled_at': '2026-05-26T13:24:49.378406+00:00', 'source_type': 'csv_admission', 'content_hash': 'b35dc1385442c727cf2cf0ece57f60cedba704ef9a49f98228ea3b1039ca544a'}
content: [검색 키워드]
학과: AI컴퓨팅학과
문서유형: admission
데이터출처유형: csv_admission
제목: 석사과정
섹션: 지원 자격
입학항목: eligibility
출처URL: https://aic.kaist.ac.kr/#/admission-grad

[문서 내용]
[AI컴퓨팅학과 입학 정보]
항목: eligibility
페이지 제목: 대학원과정 입학 안내
섹션: 지원 자격
제목: 석사과정
내용: 